In [1]:
import sys
# Install missing integration packages along with core requirements
!{sys.executable} -m pip install --no-cache-dir --prefer-binary \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    sentence-transformers \
    chromadb \
    pypdf \
    tqdm \
    groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 206.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 246.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 292.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 261.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 284.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 206.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

In [2]:
import importlib
import sys
importlib.invalidate_caches()

try:
    import langchain_huggingface
    import langchain_chroma
    import langchain_text_splitters
    import torch
    print(f"Success: All integration libraries are available.")
    print(f"GPU available: {torch.cuda.is_available()}")
except ImportError as e:
    print(f"Import failed: {e}.")

Success: All integration libraries are available.
GPU available: True


In [3]:
import sys
try:
    import langchain_huggingface
    import langchain_chroma
    import torch
    print("✅ Environment is fully synced")
except ImportError:
    print("❌ Packages not found")

✅ Environment is fully synced. You can now run the RAG pipeline cell (a7a858cf).


In [16]:
import os
import re
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from tqdm.auto import tqdm
import torch

# Configure Groq API
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name="llama-3.1-8b-instant")
device = "cuda" if torch.cuda.is_available() else "cpu"

file_path = "/content/Seerat e Mustafa_new.pdf"
if not os.path.exists(file_path):
    print(f"Error: File {file_path} not found.")
    vectorstore = None
else:
    print("Loading and classifying into Chapters...")
    loader = PyPDFLoader(file_path)
    pages = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    docs = text_splitter.split_documents(pages)

    # Refined Granular Chapter Mapping rules
    # We prioritize specific battles over the general expedition tag
    chapter_rules = [
        (r"Genealogy|Ancestry|Birth|Childhood", "Early Life and Ancestry"),
        (r"Prophethood|First Revelation|Nubuwwat", "Call to Prophethood"),
        (r"Migration|Hijrat|Madinah", "The Hijrah to Madinah"),
        # Granular Expeditions
        (r"Battle of Badr|Ghazwa-e-Badr", "Expedition: Badr"),
        (r"Battle of Uhud|Ghazwa-e-Uhud", "Expedition: Uhud"),
        (r"Khandaq|Trench|Ahzab", "Expedition: The Trench"),
        (r"Khaibar|Khaybar", "Expedition: Khaibar"),
        (r"Hunain", "Expedition: Hunain"),
        (r"Badr|Uhud|Khandaq|Khaibar|Hunain|Military", "Major Military Expeditions"),
        (r"Treaty|Hudaibiyah|Makkah Conquest", "Diplomacy and Triumph"),
        (r"Farewell|Demise|Death|Wafaat", "The Final Days and Legacy")
    ]

    current_chapter = "Introduction and Background"
    for doc in docs:
        content = doc.page_content
        for pattern, chapter_name in chapter_rules:
            if re.search(pattern, content, re.IGNORECASE):
                current_chapter = chapter_name
                break
        doc.metadata["chapter"] = current_chapter
        doc.metadata["source"] = "Seerat e Mustafa"

    hf_embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": device}
    )

    # Re-initialize vector store with granular metadata
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=hf_embeddings,
        collection_name="granular_chapter_rag_v3"
    )
    print("✅ Vector store ready with Granular Sub-Chapter divisions.")

Loading and re-classifying into Granular Sub-Chapters...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector store ready with Granular Sub-Chapter divisions.


In [18]:
if 'vectorstore' in globals() and vectorstore is not None:
    test_query = "Details of the Battle of Badr: strategy and outcome."
    # Target the specific sub-chapter to prevent cross-battle contamination
    target_chapter = "Expedition: Badr"
    chapter_filter = {"chapter": target_chapter}
    K_VALUE = 10

    print(f"--- MULTI-STRATEGY EVALUATION (k={K_VALUE}) ---")
    print(f"Filtering for: {target_chapter}\n")

    # 1. Baseline Retrieval
    print(f"[Strategy 1] Baseline Retrieval")
    baseline_docs = vectorstore.similarity_search(test_query, k=K_VALUE, filter=chapter_filter)
    baseline_final = rag.get_answer(test_query, baseline_docs)
    print(f"Baseline Result:\n{baseline_final}\n")
    print("-"*30)

    # 2. HyDE within Chapter Filter
    print(f"[Strategy 2] HyDE Retrieval")
    hyde_msg = rag.hyde_prompt.format_messages(question=test_query)
    hypothetical_ans = llm.invoke(hyde_msg).content
    hyde_docs = vectorstore.similarity_search(hypothetical_ans, k=K_VALUE, filter=chapter_filter)
    hyde_final = rag.get_answer(test_query, hyde_docs)
    print(f"HyDE Result:\n{hyde_final}\n")
    print("-"*30)

    # 3. Sub-query Decomposition (Previous Winner)
    print(f"[Strategy 3] Sub-query Decomposition")
    sub_msg = "Break this down into 2 distinct sub-questions for: " + test_query
    try:
        sub_resp = eval_llm.invoke(sub_msg).content
    except NameError:
        sub_resp = llm.invoke(sub_msg).content

    sqs = [q.strip() for q in sub_resp.split('\n') if '?' in q][:2]

    sub_docs = []
    for sq in sqs:
        print(f"  -> Searching Sub-query (k=5): {sq}")
        sub_docs.extend(vectorstore.similarity_search(sq, k=5, filter=chapter_filter))

    sub_final = rag.get_answer(test_query, sub_docs)
    print(f"\nSub-query Result:\n{sub_final}")
else:
    print("Vectorstore not initialized.")

--- MULTI-STRATEGY EVALUATION (k=10) ---
Filtering for: Expedition: Badr

[Strategy 1] Baseline Retrieval
Baseline Result:
Based on the provided text, here are the details of the Battle of Badr:

**Preparation for the Battle:**

- Rasulullah  and the Sahaabah  landed at Badr.
- The disbelievers had already seized control of the water springs and better areas of Badr.
- The Muslims were left with no water and unsuitable areas.
- Rasulullah  instructed his companions to reserve their arrows for when the disbelieving mob rushed upon them.

**The Battle:**

- After the deaths of ‘Utbah and Shaybah, the battle kicked off in earnest.
- Rasulullah  emerged from his hut and laid out the troops.

**Key Events:**

- Rasulullah  offered two Rakaat of Salaah in his hut with Hadhrat Abu Bakr .
- S‘ad bin Mu’aaz  stood guard at the door wielding a sword.
- Rasulullah  pointed out the exact locations where each person would be slain in the morning.
- Not one of them fell beyond a hair’s bread